In [1]:
!python --version

Python 3.11.14


### Get OWL files

https://reactome.org/download/current/biopax/{pathway_id}_level3.owl

In [35]:
import sys
from pathlib import Path

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.dashcyto_lib import DASH_CYTO
from libs.reactome_lib import Reactome


ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
root_reactome = ROOT0 / 'data/reactome'

rea = Reactome(root_reactome=root_reactome)

In [4]:
dfr = rea.open_reactome()
print(len(dfr))
dfr.head(3)

2119


,pathway,pathway_id
0,2-LTR Circle Formation,R-HSA-164843
1,Abacavir ADME,R-HSA-2161522
2,Abacavir Transmembrane Transport,R-HSA-2161517


In [ ]:
force=False

i=0
pathway_id = dfr.iloc[0].pathway_id

print(f"Downloading {pathway_id}")

dic = rea.download_reactome_owl(pathway_id=pathway_id, force=force, timeout=60)
    
print(dic)

In [ ]:
for i, row in dfr.iterrows():
    pathway_id = row.pathway_id
    print(f"Downloading {pathway_id}")

    dic = rea.download_reactome_owl(pathway_id=pathway_id, force=force, timeout=60)
    print(dic['status'], dic['msg'])

    if i==3: break

### pybiopax

uv pip install pybiopax  
uv pip install ipywidgets  

In [10]:
from pybiopax import model_from_owl_file
from collections import Counter

In [8]:
os.listdir(rea.root_reactome)

['gene_association.reactome.gz',
 'owl',
 'reactome_summary.tsv',
 'ReactomePathwaysRelation.tsv',
 'Homo_sapiens.owl',
 'Pathways2GoTerms_human.tsv',
 'reactome_pathways_abstract.tsv',
 'reactome_pathways_species.tsv',
 'reactome_pathways.tsv',
 'reactome_pathways_human_gmt.tsv',
 'ReactomePathways_GeneSet.gmt.zip',
 'homo_sapiens_reactions.3.1.sbml.tgz',
 'reactome.graphdb.tgz',
 'homo_sapiens.sbgn.tar.gz',
 'all_species.3.1.sbml.tgz',
 'Reactions2GoTerms_human.tsv',
 'biopax.zip',
 'papers',
 'pathway_json',
 'Reactome_Pathways_2024.tsv',
 'ReactomePathways.tsv',
 'reactome_pathways_human.tsv']

In [9]:
fname = "Homo_sapiens.owl"
filename = rea.root_reactome / fname
model = model_from_owl_file(str(filename))

print(type(model))
print(len(model.objects))



Processing OWL elements:   0%|          | 0.00/545k [00:00<?, ?it/s]

<class 'pybiopax.biopax.model.BioPaxModel'>
544590


In [12]:
class_counts = Counter(type(obj).__name__ for obj in model.objects.values())

i=0
for cls, n in class_counts.most_common(20):
    print(cls, n)
    i+=1
    #if i==10: break

UnificationXref 181879
SequenceSite 67532
Stoichiometry 43907
PublicationXref 39028
Protein 33650
FragmentFeature 30043
SequenceInterval 30043
PathwayStep 18863
Complex 17459
BiochemicalReaction 15994
RelationshipXref 14498
ProteinReference 12204
ModificationFeature 11434
Catalysis 6800
SmallMolecule 5461
Control 3206
SmallMoleculeReference 3063
Pathway 2870
PhysicalEntity 2149
Dna 1579


In [27]:
def get_reactome_ids_given_obj2(obj):
    ids = set()

    uid = getattr(obj, "uid", None)
    if uid and "R-HSA-" in uid:
        ids.add(uid.split("/")[-1].replace("#", ""))

    xrefs = getattr(obj, "xref", []) or []

    for x in xrefs:
        xid = getattr(x, "id", None)
        db = getattr(x, "db", None)

        if xid and str(xid).startswith("R-HSA-"):
            ids.add(str(xid))

        if db and xid and "Reactome" in str(db):
            ids.add(str(xid))

    return ids

def get_reactome_ids_given_obj(obj):
    ids = set()

    # Sometimes stable IDs appear in uid
    uid = getattr(obj, "uid", None)
    if isinstance(uid, str) and uid.startswith("R-HSA-"):
        ids.add(uid)

    # Usually stable IDs are in xref
    for x in getattr(obj, "xref", []) or []:
        xid = getattr(x, "id", None)
        db = getattr(x, "db", None)

        if xid and str(xid).startswith("R-HSA-"):
            ids.add(str(xid))

        if db and "Reactome" in str(db) and xid:
            ids.add(str(xid))

    return ids

def get_pathways_from_model(model):
    
    pathways = [ obj for obj in model.objects.values()  if type(obj).__name__ == "Pathway"  ]

    pathway_index = {}

    for p in pathways:
        ids = get_reactome_ids_given_obj(p)

        for rid in ids:
            pathway_index[rid] = p

    return pathways, pathway_index

pathways, pathway_index = get_pathways_from_model(model)
print("Number of pathways:", len(pathways))
print("Number of pathway IDs:", len(pathway_index))


Number of pathways: 2870
Number of pathway IDs: 5740


In [43]:
dic={}
icount=-1

for p in pathways:
    ids = get_reactome_ids_given_obj(p)

    for rid in ids:
        icount += 1
        dic[icount] = {}

        dic2 = dic[icount]

        dic2['pathway_object'] = p
        dic2['rid'] = rid
        dic2['is_ID'] = str(rid).startswith('R-HSA-')

dfo = pd.DataFrame.from_dict(dic, orient='index')
dfo.head(12)

,pathway_object,rid,is_ID
0,<pybiopax.biopax.base.Pathway object at 0x74d8...,R-HSA-1852241,True
1,<pybiopax.biopax.base.Pathway object at 0x74d8...,1852241,False
2,<pybiopax.biopax.base.Pathway object at 0x74d8...,1592230,False
3,<pybiopax.biopax.base.Pathway object at 0x74d8...,R-HSA-1592230,True
4,<pybiopax.biopax.base.Pathway object at 0x74d8...,2151209,False
5,<pybiopax.biopax.base.Pathway object at 0x74d8...,R-HSA-2151209,True
6,<pybiopax.biopax.base.Pathway object at 0x74d8...,R-HSA-2151201,True
7,<pybiopax.biopax.base.Pathway object at 0x74d8...,2151201,False
8,<pybiopax.biopax.base.Pathway object at 0x74d8...,R-HSA-8949613,True
9,<pybiopax.biopax.base.Pathway object at 0x74d8...,8949613,False


In [17]:
for p in pathways[:10]:
    print(p.uid, getattr(p, "display_name", None), getattr(p, "pid", None)) 

Pathway1 Organelle biogenesis and maintenance None
Pathway2 Mitochondrial biogenesis None
Pathway3 Activation of PPARGC1A (PGC-1alpha) by phosphorylation None
Pathway4 Transcriptional activation of mitochondrial biogenesis None
Pathway5 Cristae formation None
Pathway6 Cilium Assembly None
Pathway7 Assembly of the 9+0 primary cilium None
Pathway8 Anchoring of the basal body to the plasma membrane None
Pathway9 Cargo trafficking to the periciliary membrane None
Pathway10 VxPx cargo-targeting to cilium None


In [25]:
p.__dict__

{'_controller_of': set(),
 'xref': [<pybiopax.biopax.util.UnificationXref at 0x74d824d83dd0>,
 'display_name': 'VxPx cargo-targeting to cilium',
 'standard_name': None,
 '_name': [],
 'evidence': [],
 'uid': 'Pathway10',
 'comment': ['A number of membrane proteins destined for the ciliary membrane are recognized by ARF4 in the trans-Golgi network, initiating the formation of a ciliary targeting complex that directs the passage of these cargo to the cilium (Mazelova et al, 2009; Geng et al, 2006; Jenkins et al, 2006; Ward et al, 2011; reviewed in Deretic, 2013). Although there is some support for the presence of a VxPx or related motif in the C-terminal tail of cargo destined for ARF4-mediated transport to the cilium, the details of this  have not been definitively established and other ciliary targeting sequences have also been identified (reviewed in Deretic, 2013; Bhogaraju et al, 2013).',
  'Authored: Rothfels, Karen, 2014-08-22',
  'Reviewed: Lorentzen, Esben, 2014-11-10',
  'Revie

In [23]:
i=0
for key, pathw in pathway_index.items():
    print(f"{key:15} {pathw}  {pathw.uid} {pathw.display_name}")
    i+=1
    if i==10: break

R-HSA-1852241   <pybiopax.biopax.base.Pathway object at 0x74d825361b10>  Pathway1 Organelle biogenesis and maintenance
1852241         <pybiopax.biopax.base.Pathway object at 0x74d825361b10>  Pathway1 Organelle biogenesis and maintenance
1592230         <pybiopax.biopax.base.Pathway object at 0x74d825362410>  Pathway2 Mitochondrial biogenesis
R-HSA-1592230   <pybiopax.biopax.base.Pathway object at 0x74d825362410>  Pathway2 Mitochondrial biogenesis
2151209         <pybiopax.biopax.base.Pathway object at 0x74d825362710>  Pathway3 Activation of PPARGC1A (PGC-1alpha) by phosphorylation
R-HSA-2151209   <pybiopax.biopax.base.Pathway object at 0x74d825362710>  Pathway3 Activation of PPARGC1A (PGC-1alpha) by phosphorylation
R-HSA-2151201   <pybiopax.biopax.base.Pathway object at 0x74d8253e7790>  Pathway4 Transcriptional activation of mitochondrial biogenesis
2151201         <pybiopax.biopax.base.Pathway object at 0x74d8253e7790>  Pathway4 Transcriptional activation of mitochondrial biogenesis


In [16]:
pid = "R-HSA-164843"

p = pathway_index.get(pid)

if p is None:
    print("Not found")
else:
    print("Found:", p.display_name)

Found: 2-LTR circle formation


### Recursively collect all referenced objects

In [28]:
from collections import deque


def is_biopax_object(x):
    return hasattr(x, "uid") and hasattr(x, "__dict__")


def collect_referenced_objects(root_obj, include_inverse=False):
    """
    Recursively collect all BioPAX objects reachable from root_obj.

    include_inverse=False is safer because inverse links such as:
        _participant_of
        _controller_of
    may pull in too much of the full model.

    For rebuilding one pathway OWL, usually use forward references only.
    """

    dic_seen = {}
    queue = deque([root_obj])

    while queue:
        obj = queue.popleft()

        uid = getattr(obj, "uid", None)
        if uid is None:
            continue

        if uid in dic_seen:
            continue

        dic_seen[uid] = obj

        for attr, value in obj.__dict__.items():

            # Skip inverse/back-reference fields unless explicitly requested
            if not include_inverse and attr.startswith("_"):
                continue

            if value is None:
                continue

            if is_biopax_object(value):
                queue.append(value)

            elif isinstance(value, (list, tuple, set)):
                for item in value:
                    if is_biopax_object(item):
                        queue.append(item)

            elif isinstance(value, dict):
                for item in value.values():
                    if is_biopax_object(item):
                        queue.append(item)

    return dic_seen

In [29]:
i=0
p = pathways[i]
print(f"Pathway {i}: {p.uid} - {p.display_name}")

if p.uid is None:
    print("Pathway has no stable ID")
else:
    sub_objects = collect_referenced_objects(p, include_inverse=False)

    print("Objects in pathway subset:", len(sub_objects))

Pathway 0: Pathway1 - Organelle biogenesis and maintenance
Objects in pathway subset: 62748


In [31]:
from pybiopax.biopax import BioPaxModel

sub_model = BioPaxModel(objects=sub_objects)

### Next check is the OWL export function available in your installed pybiopax:

In [32]:
import pybiopax

print([x for x in dir(pybiopax) if "owl" in x.lower()])

['model_from_owl_file', 'model_from_owl_gz', 'model_from_owl_str', 'model_from_owl_url', 'model_to_owl_file', 'model_to_owl_str']


In [62]:
dcy = DASH_CYTO(ROOT0=ROOT0)

def get_pathway_names(p):

    dfa = dfo[
        (dfo["pathway_object"].map(lambda x: x is p)) &
        (dfo["is_ID"])
    ]

    if dfa.empty:
        pathway_id = ""
    else:
        row = dfa.iloc[0]
        pathway_id = row.rid

    uid = p.uid
    pathway = p.display_name
    
    return uid, pathway_id, pathway

uid, pathway_id, pathway = get_pathway_names(p)


In [63]:
fname_owl = f"{pathway_id}_level3.owl"

filename = dcy.root_owl / fname_owl

filename

PosixPath('/home/flavio/uv/perturb_agent/owl/R-HSA-9843970_level3.owl')

In [64]:


pybiopax.model_to_owl_file(sub_model, filename)

Serializing OWL elements:   0%|          | 0.00/62.7k [00:00<?, ?it/s]

In [65]:
test_model = pybiopax.model_from_owl_file(filename)

print(type(test_model))
print(len(test_model.objects))

Processing OWL elements:   0%|          | 0.00/62.7k [00:00<?, ?it/s]

<class 'pybiopax.biopax.model.BioPaxModel'>
62748


### DASH_CYTO

In [37]:
from rdflib import RDF, Graph, Namespace


In [44]:
dfo.head(2)

,pathway_object,rid,is_ID
0,<pybiopax.biopax.base.Pathway object at 0x74d8...,R-HSA-1852241,True
1,<pybiopax.biopax.base.Pathway object at 0x74d8...,1852241,False


In [46]:
p

In [49]:
type(p)

pybiopax.biopax.base.Pathway

In [50]:
type(dfo.pathway_object.iloc[0])

pybiopax.biopax.base.Pathway

In [58]:
dfa = dfo[
    (dfo["pathway_object"].map(lambda x: x is p)) &
    (dfo["is_ID"])
]
dfa

,pathway_object,rid,is_ID
5738,<pybiopax.biopax.base.Pathway object at 0x74d7...,R-HSA-9843970,True


In [66]:
rdf = dcy.read_owl(pathway_id, pathway)
height = "1100px"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop)

Open in browser: http://localhost:8050
Dash app running on http://127.0.0.1:8050/


In [40]:
print(f"Pathway {i}: {p.uid} - {p.display_name}")

Pathway 0: Pathway1 - Organelle biogenesis and maintenance
